# Notebook to demo GWR modeling.

In [1]:
import warnings

# warnings.simplefilter(action='ignore', category=FutureWarning)
# warnings.filterwarnings("ignore")

In [2]:
from collections import OrderedDict
from typing import Tuple

import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from itertools import product
from statsmodels.regression.linear_model import WLS

import geofunctions as S

In [10]:
warnings.simplefilter(action='ignore', category=FutureWarning)
spark.version

'3.3.2'

In [4]:
xmin, ymin, xmax, ymax = (
    35.46417853366222,
    33.841706023612225,
    35.57065336261585,
    33.906762501363865,
)

xdel = xmax - xmin
ydel = ymax - ymin

cell1 = min(xdel, ydel) / 10.0
cell2 = cell1 * 0.5

### Create dataframe with geom, dep var and ind vars.

In [5]:
df = (
    spark
    .range(1000)
    .drop("id")
    .withColumn("x", F.rand() * xdel + xmin)
    .withColumn("y", F.rand() * ydel + ymin)
    # Define dep attribute
    .withColumn("y0", F.rand() * 100)
    # define ind attributes
    .withColumn("x1", F.rand() * 100)
    .withColumn("x2", F.rand() * 100)
)

In [6]:
df = (df
      .withColumn("q", S.st_xtoq("x", cell1))
      .withColumn("r", S.st_ytor("y", cell1))
     )

### Define neighborhood range.

In [7]:
data = list(product(range(-1, 2), range(-1, 2)))
prod = spark.createDataFrame(data, "i int,j int")

In [8]:
def gwr(qr:Tuple[float,float], df:pd.DataFrame) -> pd.DataFrame:
    q, r = qr
    # calculate center
    x0 = q * cell1 + cell2
    y0 = r * cell1 + cell2
    dx = df.x - x0
    dy = df.y - y0
    # Get weights based on distance.
    w = np.clip(1.0 - np.sqrt(dx * dx + dy * dy) / cell1, 0.0, 1.0)
    if np.min(w) < np.max(w):
        model = WLS(df["y0"], df[["x1", "x2"]], w)
        res = model.fit()
        data = [
            ("q", [q]),
            ("r", [r]),
            ("n", [len(df)]),
            ("p0", [res.params[0]]),
            ("p1", [res.params[1]]),
            ("r2", [res.rsquared]),
        ]
    else:
        data = [
            ("q", [q]),
            ("r", [r]),
            ("n", [0]),
            ("p0", [0.0]),
            ("p1", [0.0]),
            ("r2", [-1.0]),
        ]
    return pd.DataFrame(OrderedDict(data))

In [9]:
with pd.option_context('mode.chained_assignment', None):
    (
        df
        .crossJoin(F.broadcast(prod))
        .withColumn("q", F.col("q") + F.col("i"))
        .withColumn("r", F.col("r") + F.col("j"))
        .drop("i", "j")
        .groupBy("q", "r")
        .applyInPandas(
            gwr, schema="q long, r long, n int, p0 double, p1 double, r2 double"
        )
        .show()
    )

/Users/mraad/spark-3.3.2-bin-hadoop3/python/lib/pyspark.zip/pyspark/sql/pandas/serializers.py:227: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
/var/folders/f3/hjs5dgkn40vcq4g_jn31nsfm0000gp/T/ipykernel_32666/2422527438.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/f3/hjs5dgkn40vcq4g_jn31nsfm0000gp/T/ipykernel_32666/2422527438.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/Users/mraad/spark-3.3.2-bin-hadoop3/python/lib/pyspark.zip/pyspark/sql/pandas/serializers.py:227: FutureWarning: is_categorical_dty

+----+----+---+--------------------+--------------------+------------------+
|   q|   r|  n|                  p0|                  p1|                r2|
+----+----+---+--------------------+--------------------+------------------+
|5450|5200|  0|                 0.0|                 0.0|              -1.0|
|5450|5201|  4| 0.45326638683328685|  0.4653304616155546|               1.0|
|5450|5202|  5| 0.45326638683328696|  0.4653304616155546|               1.0|
|5450|5203|  7|  0.9344091254152421|0.016385513272452815|               1.0|
|5450|5204|  9|  0.6201772255347326| 0.45476381253478776|               1.0|
|5450|5205|  0|                 0.0|                 0.0|              -1.0|
|5450|5206|  0|                 0.0|                 0.0|              -1.0|
|5450|5207|  0|                 0.0|                 0.0|              -1.0|
|5450|5208| 20|  0.9996233919582126|   -0.64105515362514|0.9952038224635261|
|5450|5209| 19|  0.5773871649182843| 0.23930275513372495|0.6010429124463588|

/var/folders/f3/hjs5dgkn40vcq4g_jn31nsfm0000gp/T/ipykernel_32666/2422527438.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/f3/hjs5dgkn40vcq4g_jn31nsfm0000gp/T/ipykernel_32666/2422527438.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/Users/mraad/spark-3.3.2-bin-hadoop3/python/lib/pyspark.zip/pyspark/sql/pandas/serializers.py:227: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
/var/folders/f3/hjs5dgkn40vcq4g_jn31nsfm0000gp/T/ipykernel_32666/2422527438.py:17: FutureWarning: Series.__getitem__ treating keys a